In [2]:
! pip install pinecone

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------- ----------- 1.8/2.6 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 10.3 MB/s  0:00:00

   ---------------------------------------- 0/5 [msgspec]
   -------- ------------------------------- 1/5 [hyperframe]
   ---------------- ----------------------- 2/5 [hpack]
   ------------------------ --------------- 3/5 [h2]
   ------------------------ --------------- 3/5 [h2]
   ------------------------ --------------- 3/5 [h2]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -----------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install pinecone openai pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import os

In [5]:
pinecone_api_key ="pcsk_2pjurJ_KvB9WzTus3jKhe6UPbEomLkAHMyUJEdmRkq7ercBR1pN9eFmaFRNCTWsbvfxqnq"

In [6]:
index_name = "retail-aqueries"

In [8]:
model = SentenceTransformer('all-MiniLM-L6-V2')
model.get_sentence_embedding_dimension()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5326.33it/s]
C:\Users\Jaya krishna\AppData\Local\Temp\ipykernel_19760\3653118381.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


384

In [9]:
pc = Pinecone(api_key=pinecone_api_key)
existing_indexes = [index["name"]for index in pc.list_indexes()]

In [10]:
existing_indexes

['resume-ranking', 'retail-queries']

In [11]:
existing_indexes = [index["name"] for index in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(name=index_name,dimension=384,metric="cosine",
                    spec=ServerlessSpec(cloud = "aws",region="us-east-1"))
index=pc.Index(index_name)

In [12]:
retail_aqueries = [
    {
        "query": "Show me men's running shoes under ₹3000",
        "category": "Men's footwear",
        "attributes": "running shoes, price below ₹3000"
    },
    {
        "query": "I need a black formal shirt for office",
        "category": "Men's clothing",
        "attributes": "formal shirt, black, office"
    },
    {
        "query": "Find laptops with at least 16GB RAM",
        "category": "Electronics",
        "attributes": "laptop, RAM >= 16GB"
    },
    {
        "query": "Show me smartphones under ₹20000 with a good camera",
        "category": "Electronics",
        "attributes": "smartphone, price below ₹20000, good camera"
    },
    {
        "query": "I want a red dress for women",
        "category": "Women's clothing",
        "attributes": "dress, red"
    },
    {
        "query": "Find wireless headphones with noise cancellation",
        "category": "Electronics",
        "attributes": "wireless headphones, noise cancellation"
    },
    {
        "query": "Show me cotton t-shirts below ₹1000",
        "category": "Men's clothing",
        "attributes": "cotton t-shirt, price below ₹1000"
    },
    {
        "query": "I need a waterproof backpack for college",
        "category": "Bags",
        "attributes": "backpack, waterproof, college"
    },
    {
        "query": "Find running shoes for women",
        "category": "Women's footwear",
        "attributes": "running shoes"
    },
    {
        "query": "Show me 55 inch 4K smart TVs",
        "category": "Electronics",
        "attributes": "TV, 55 inch, 4K, smart"
    },
    {
        "query": "I need a microwave oven under ₹12000",
        "category": "Home appliances",
        "attributes": "microwave oven, price below ₹12000"
    },
    {
        "query": "Find skincare products for dry skin",
        "category": "Beauty",
        "attributes": "skincare, dry skin"
    },
    {
        "query": "Show me men's watches under ₹5000",
        "category": "Accessories",
        "attributes": "men's watch, price below ₹5000"
    },
    {
        "query": "I need a lightweight laptop for programming",
        "category": "Electronics",
        "attributes": "laptop, lightweight, programming"
    },
    {
        "query": "Find kitchen mixer grinders below ₹4000",
        "category": "Kitchen appliances",
        "attributes": "mixer grinder, price below ₹4000"
    },
    {
        "query": "Show me children's school bags",
        "category": "Kids",
        "attributes": "school bag, children"
    },
    {
        "query": "I want Bluetooth speakers with long battery life",
        "category": "Electronics",
        "attributes": "Bluetooth speaker, long battery"
    },
    {
        "query": "Find comfortable jeans for men under ₹2000",
        "category": "Men's clothing",
        "attributes": "jeans, comfortable, price below ₹2000"
    },
    {
        "query": "Show me face wash for oily skin",
        "category": "Beauty",
        "attributes": "face wash, oily skin"
    },
    {
        "query": "I need a fitness smartwatch with heart-rate monitoring",
        "category": "Electronics",
        "attributes": "fitness smartwatch, heart-rate monitoring"
    }
]

In [16]:
vectors = []
for i , retail_aqueries in enumerate(retail_aqueries):
    embedding = model.encode(
        retail_aqueries["query"]
    ).tolist()

    vectors.append({
        "id":str(i),
        "values":embedding,
        "metadata":{
            "name":retail_aqueries["query"],
            "retail_aqueries_text":retail_aqueries["query"]
        }
    })

In [18]:
index.upsert(vectors=vectors)

UpsertResponse(upserted_count=20)

In [22]:
request = " I need a lightweight laptop for programming"

In [23]:
query_embedding = model.encode(
    request
).tolist()

In [26]:
results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True)

In [27]:
print("\n")
print("="* 70)
print("          TOP 3 MATCHING RESUMES")
print("="* 70)

for rank, result in enumerate(results["matches"][:3],start=1):
    print(f"\nRank          : {rank}")
    print(f"Name    : {result['metadata']['name']}")
    print(f"Query ID    : {result['id']}")
    print(f"Similarity  :  {result['score']:.4f}")

    print("-" * 70)



          TOP 3 MATCHING RESUMES

Rank          : 1
Name    : I need a lightweight laptop for programming
Query ID    : 13
Similarity  :  0.9994
----------------------------------------------------------------------

Rank          : 2
Name    : Find laptops with at least 16GB RAM
Query ID    : 2
Similarity  :  0.4751
----------------------------------------------------------------------

Rank          : 3
Name    : I need a waterproof backpack for college
Query ID    : 7
Similarity  :  0.3259
----------------------------------------------------------------------


In [28]:
request

' I need a lightweight laptop for programming'